In [15]:
import re
import os
import sys
import pandas as pd
from datetime import *

In [16]:
# This script will edit a stardard XML to generate a targetedBeast XML

In [17]:
# Generate initial tree block

def generate_init_block(TREE):

    line1 = "		<init spec='targetedbeast.init.ensureNoZeroEdgeLength' id='UPGMATree.t:XYZ26'"
    line2 = "          initial=\"@Tree.t:insert_tree\" clusterType='upgma' taxa='@insert_tree'/>"

    line2 = line2.replace("insert_tree", str(TREE))

    block = "\n".join([line1, line2])

    return(block)

In [18]:
# Generate consensusWeights block

def generate_consensusweights(TREE):

    line1 = "                <distribution id=\"consensusWeights\" spec=\"targetedbeast.edgeweights.ParsimonyWeights\" tree=\"@Tree.t:insert_tree\" maxWeight=\"20\">"
    line2 = "					<data spec=\"targetedbeast.alignment.ConsensusAlignment\" data=\"@insert_tree\"/>"
    line3 = "				</distribution>"

    line1 = line1.replace("insert_tree", str(TREE))
    line2 = line2.replace("insert_tree", str(TREE))

    block = "\n".join([line1, line2, line3])

    return(block)

In [19]:
# Generate new operators

def generate_targeted_operators(TREE):

    line1 = "        <operator id=\"WeightBasedNodeRandomizer\" spec=\"targetedbeast.operators.WeightBasedNodeRandomizer\" percentage=\"0.01\" optimise=\"true\" tree=\"@Tree.t:insert_tree\" weight=\"0.5\" edgeWeights=\"@consensusWeights\"/>"
    line2 = "        <operator id=\"HeightBasedNodeRandomizer\" spec=\"targetedbeast.operators.HeightBasedNodeRandomizer\" percentage=\"0.01\" optimise=\"true\" tree=\"@Tree.t:insert_tree\" weight=\"0.5\"/>"
    line3 = "        <operator id=\"CoalescentConstantSubtreeSlide.t:insert_tree\" spec=\"kernel.BactrianSubtreeSlide\" tree=\"@Tree.t:insert_tree\" weight=\"20.0\"/>"
    line4 = "		<operator id=\"Range\" spec=\"targetedbeast.operators.RangeSlide\" tree=\"@Tree.t:insert_tree\" weight=\"30.0\" size=\"0.1\" edgeWeights=\"@consensusWeights\"/>"
    line5 = "        <operator id=\"UnTargetedWide\" spec=\"targetedbeast.operators.WeightedWideOperator\" tree=\"@Tree.t:insert_tree\" weight=\"5.0\"/>"
    line6 = "        <operator id=\"TargetedWide\" spec=\"targetedbeast.operators.WeightedWideOperator\" tree=\"@Tree.t:insert_tree\" weight=\"15.0\" edgeWeights=\"@consensusWeights\"/>"
    line7 = "        <operator id=\"TargetedWilsonBalding\" spec=\"targetedbeast.operators.TargetedWilsonBalding\" tree=\"@Tree.t:insert_tree\" weight=\"5\" edgeWeights=\"@consensusWeights\"/>"
    line8 = "        <operator id=\"TargetedWilsonBalding2\" spec=\"targetedbeast.operators.TargetedWilsonBalding\" tree=\"@Tree.t:insert_tree\" weight=\"1\" edgeWeights=\"@consensusWeights\" useEdgeLength=\"true\"/>"
    line9 = "        <operator id=\"ScaleAll\" spec=\"targetedbeast.operators.IntervalScaleOperator\" tree=\"@Tree.t:insert_tree\" weight=\"0.25\" scaleFactor=\"0.9\">"
    line10 = "			<down idref=\"clockRate.c:insert_tree\"/>"
    line11 = "		</operator>"
    line12 = "        <operator id=\"ScaleIntervalsRandomly\" spec=\"targetedbeast.operators.IntervalScaleOperator\" tree=\"@Tree.t:insert_tree\" weight=\"0.25\" scaleFactor=\"0.9\" scaleAllNodesIndependently=\"true\"/>"

    line1 = line1.replace("insert_tree", str(TREE))
    line2 = line2.replace("insert_tree", str(TREE))
    line3 = line3.replace("insert_tree", str(TREE))
    line4 = line4.replace("insert_tree", str(TREE))
    line5 = line5.replace("insert_tree", str(TREE))
    line6 = line6.replace("insert_tree", str(TREE))
    line7 = line7.replace("insert_tree", str(TREE))
    line8 = line8.replace("insert_tree", str(TREE))
    line9 = line9.replace("insert_tree", str(TREE))
    line10 = line10.replace("insert_tree", str(TREE))
    line12 = line12.replace("insert_tree", str(TREE))

    block = line1 + "\n" + line2 + "\n\n" + line3 + "\n" + line4 + "\n\n" + line5 + "\n" + line6 + "\n" + line7 + "\n" + line8 + "\n\n" + line9 + "\n" + line10 + "\n" + line11 + "\n" + line12 + "\n"

    return(block)

In [20]:
def write_xml(OUTPUT, INPUT, init_block, consensusweights_block, targetedoperators_block): 

    with open(OUTPUT, "w") as outfile: 
        outfile.write("")
    
    with open(INPUT, "r") as infile: 
        for line in infile:

            # Change the init block
            if "<init id=" in line: 
                with open(OUTPUT, "a") as outfile:
                    outfile.write("\n" + init_block + "\n" + "<!--" + "\n" + line)

            elif "</init>" in line:
                with open(OUTPUT, "a") as outfile:
                    outfile.write(line + "\n" + "-->" + "\n")

            # Add in consensusWeights
            elif "<distribution id=\"prior\" spec=\"CompoundDistribution\">" in line:
                with open(OUTPUT, "a") as outfile:
                    outfile.write(line + "\n" + consensusweights_block + "\n\n")

            # Comment out the tree transformation
            elif "<f idref=\"Tree.t" in line:
                with open(OUTPUT, "a") as outfile:
                    outfile.write("<!--" + "\n" + line + "-->" + "\n")

            # I COULD NOT FIGURE OUT HOW TO COMMENT OUT THE UPDOWN OPERATORS SO REMEMBER TO DO THAT BY HAND
            # if "<operator id=\"strictClockUpDownOperator.c:" in line:
            #     with open(OUTPUT, "a" as outfile:
            #         outfile.write("<!--" + "\n" + line)

            # if "<down idref

            # # Comment out old operators and add in targetedBeast operators (skygrid)
            # elif "<operator id=\"SkyGridBICEPSEpochTop.t:" in line:
            #     with open(OUTPUT, "a") as outfile:
            #         outfile.write("<!--" + "\n" + line)

            # elif "<operator id=\"SkyGridWilsonBalding.t:" in line:
            #     with open(OUTPUT, "a") as outfile:
            #         outfile.write(line + "-->" + "\n" + targetedoperators_block + "\n")   

            # Comment out old operators and add in targetedBeast operators (exponential)
            # elif "<operator id=\"CoalescentExponentialBICEPSEpochTop.t:" in line:
            #     with open(OUTPUT, "a") as outfile:
            #         outfile.write("<!--" + "\n" + line)

            # elif "<operator id=\"CoalescentExponentialWilsonBalding.t:" in line:
            #     with open(OUTPUT, "a") as outfile:
            #         outfile.write(line + "-->" + "\n" + targetedoperators_block + "\n")

            # Comment out old operators and add in targetedBeast operators (skyline)
            # elif "<operator id=\"BayesianSkylineBICEPSEpochTop.t:" in line:
            #     with open(OUTPUT, "a") as outfile:
            #         outfile.write("<!--" + "\n" + line)

            # elif "<operator id=\"BayesianSkylineWilsonBalding.t:" in line:
            #     with open(OUTPUT, "a") as outfile:
            #         outfile.write(line + "-->" + "\n" + targetedoperators_block + "\n")

            # I think I can apply this to all demographic model operators
            elif "BICEPSEpochTop.t:" in line:
                with open(OUTPUT, "a") as outfile:
                    outfile.write("<!--" + "\n" + line)

            elif "WilsonBalding.t:" in line:
                with open(OUTPUT, "a") as outfile:
                    outfile.write(line + "-->" + "\n" + targetedoperators_block + "\n")
                        
            else: 
                with open(OUTPUT, "a") as outfile: 
                    outfile.write(line)

In [21]:
# Key assumtions made by this script: strict clock model, SkyGrid population model, no traits
# If you have a different clock, you must alter line10 of the new operators]
# If you have a different population mode, you'll have to adjust the write_xml function to comment out the correct operators
# If you have traits, add a line to the operators identical to line10 but for the trait clock

In [22]:
# Generate blocks

# tree_name = "div-tree-strains-newheader-aligned"
# tree_name = "liveharvest_aligned"
# tree_name = "prop-time-host_aligned"
# tree_name = "prop-time-loc_aligned"
# tree_name = "A1_A2_genome_newheader"
tree_name = "eq-time-loc_aligned"

init_block = generate_init_block(tree_name)
consensusweights_block = generate_consensusweights(tree_name)
targetedoperators_block = generate_targeted_operators(tree_name)

In [23]:
# INPUT = "./BEAST/empirical_trees/submission_files/2026-02-08_main-intro_equal-time-host-1000_ha_empirical_targetedbeast/inputs/main-intro_equal-time-host-1000_ha_empirical_initial.xml"
# INPUT = "./BEAST/empirical_trees/submission_files/2026-02-12_main-intro_equal-time-host-3000_ha_empirical_targetedbeast/inputs/main-intro_equal-time-host-3000_ha_empirical_initial.xml"
# INPUT = "./BEAST/empirical_trees/submission_files/2026-02-12_main-intro_equal-time-loc_ha_empirical_targetedbeast/inputs/main-intro_equal-time-loc_ha_empirical_initial.xml"
# INPUT = "./BEAST/empirical_trees/submission_files/2026-02-12_main-intro_liveharvest_ha_empirical_targetedbeast/inputs/main-intro_liveharvest_ha_empirical_initial.xml"
# INPUT = "./BEAST/empirical_trees/submission_files/2026-02-12_main-intro_prop-time-host_ha_empirical_targetedbeast/inputs/main-intro_prop-time-host_ha_empirical_initial.xml"
# INPUT = "./BEAST/empirical_trees/submission_files/2026-02-12_main-intro_prop-time-loc_ha_empirical_targetedbeast/inputs/main-intro_prop-time-loc_ha_empirical_initial.xml"
# INPUT = "../BEAST_tests/targeted_BEAST/submission_files/test_exp-model/test_C2.1_exponential_initial.xml"
# INPUT = "../BEAST_tests/targeted_BEAST/submission_files/test_skyride-model/test_C2.1_skyride_initial.xml"
# INPUT = "../BEAST_tests/targeted_BEAST/submission_files/test_skyline-model/test_C2.1_skyline_initial.xml"
# INPUT = "../H5-genotypes/BEAST/empirical_trees/submission_files/2026-02_h5_emptree_A1_A2_exponential_MLE/A1_A2_concat-gen_exponential_initial_2026-02-23.xml"
# INPUT = "../H5-genotypes/BEAST/empirical_trees/submission_files/2026-02_h5_emptree_A1_A2_skyline_MLE/A1_A2_concat-gen_skyline_initial_2026-02-23.xml"
# INPUT = "./BEAST/empirical_trees/submission_files/2026-02-26_main-intro_equal-time-loc_ha_empirical_targetedbeast_redo/inputs/main-intro_equal-time-loc_ha_empirical_initial.xml"
# INPUT = "./BEAST/empirical_trees/submission_files/2026-02-26_main-intro_prop-time-host_ha_empirical_targetedbeast_redo/inputs/main-intro_prop-time-host_ha_empirical_initial.xml"
# INPUT = "../BEAST_tests/targeted_BEAST/submission_files/test_equal-time-loc_skyline_pmacs/test_equal-time-loc_skyline_initial.xml"
# INPUT = "../BEAST_tests/targeted_BEAST/submission_files/test_equal-time-loc_constant_pmacs/test_equal-time-loc_constant_initial.xml"
INPUT = "../BEAST_tests/targeted_BEAST/submission_files/test_equal-time-loc_biceps_pmacs/test_equal-time-loc_biceps_initial.xml"

# OUTPUT = "./BEAST/empirical_trees/submission_files/2026-02-08_main-intro_equal-time-host-1000_ha_empirical_targetedbeast/test.xml"
# OUTPUT = "./BEAST/empirical_trees/submission_files/2026-02-12_main-intro_equal-time-host-3000_ha_empirical_targetedbeast/main-intro_equal-time-host-3000_ha_empirical_targeted.xml"
# OUTPUT = "./BEAST/empirical_trees/submission_files/2026-02-12_main-intro_equal-time-loc_ha_empirical_targetedbeast/main-intro_equal-time-loc_ha_empirical_targeted.xml"
# OUTPUT = "./BEAST/empirical_trees/submission_files/2026-02-12_main-intro_liveharvest_ha_empirical_targetedbeast/main-intro_liveharvest_ha_empirical_targeted.xml"
# OUTPUT = "./BEAST/empirical_trees/submission_files/2026-02-12_main-intro_prop-time-host_ha_empirical_targetedbeast/main-intro_prop-time-host_ha_empirical_targeted.xml"
# OUTPUT = "./BEAST/empirical_trees/submission_files/2026-02-12_main-intro_prop-time-loc_ha_empirical_targetedbeast/main-intro_prop-time-loc_ha_empirical_targeted.xml"
# OUTPUT = "../BEAST_tests/targeted_BEAST/submission_files/test_skyride-model/test_C2.1_skyride_targeted.xml"
# OUTPUT = "../BEAST_tests/targeted_BEAST/submission_files/test_skyline-model/test_C2.1_skyline_targeted.xml"
# OUTPUT = "../H5-genotypes/BEAST/empirical_trees/submission_files/2026-02_h5_emptree_A1_A2_skyline_MLE/A1_A2_concat-gen_skyline_targeted_2026-02-23.xml"
# OUTPUT = "./BEAST/empirical_trees/submission_files/2026-02-26_main-intro_equal-time-loc_ha_empirical_targetedbeast_redo/main-intro_equal-time-loc_ha_empirical_targeted.xml"
# OUTPUT = "./BEAST/empirical_trees/submission_files/2026-02-26_main-intro_prop-time-host_ha_empirical_targetedbeast_redo/main-intro_prop-time-host_ha_empirical_targeted.xml"
# OUTPUT = "../BEAST_tests/targeted_BEAST/submission_files/test_equal-time-loc_skyline_pmacs/test_equal-time-loc_skyline_targeted.xml"
# OUTPUT = "../BEAST_tests/targeted_BEAST/submission_files/test_equal-time-loc_constant_pmacs/test_equal-time-loc_constant_targeted.xml"
OUTPUT = "../BEAST_tests/targeted_BEAST/submission_files/test_equal-time-loc_biceps_pmacs/test_equal-time-loc_biceps_targeted.xml"

write_xml(OUTPUT, INPUT, init_block, consensusweights_block, targetedoperators_block)